# Day 09 — Mathematical Algorithms
 
**Week 2: Advanced Control Flow & Logic**
 
Topics covered: number theory (primes, GCD, LCM, factorials), iterative and recursive algorithms, modular arithmetic, digit manipulation, sequence generation, mathematical series, `math` module.
 
---

### Q1. Prime Checker
Write a function `is_prime(n)` that returns `True` if `n` is prime. Optimize it to only check divisors up to `√n` using `math.isqrt(n)`. Test it for `n = 0, 1, 2, 13, 97, 100, 999983`.

In [1]:
# Q1: Prime checker optimised with √n divisor limit

import math

def is_prime(n):
    if n < 2:                          # 0 and 1 are not prime by definition
        return False
    if n == 2:                         # 2 is the only even prime
        return True
    if n % 2 == 0:                     # all other even numbers are not prime
        return False
    for i in range(3, math.isqrt(n) + 1, 2):  # check odd divisors up to √n
        if n % i == 0:
            return False
    return True

# Test cases
test_values = [0, 1, 2, 13, 97, 100, 999983]
for n in test_values:
    print(f"is_prime({n:>7}) → {is_prime(n)}")

is_prime(      0) → False
is_prime(      1) → False
is_prime(      2) → True
is_prime(     13) → True
is_prime(     97) → True
is_prime(    100) → False
is_prime( 999983) → True


### How it works

- Any number less than 2 is not prime by definition — so we return `False` immediately.
- We handle `2` as a special case (the only even prime), then skip all other even numbers.
- For the remaining odd candidates, we only check divisors from `3` up to **√n**.
- The key insight: if `n` has a factor larger than √n, it must also have one *smaller* than √n — so we never need to go higher.
- `math.isqrt(n)` gives the integer square root without floating-point rounding errors.
- The loop step `2` means we skip even divisors entirely, roughly halving the work.

**Key takeaway:** Checking only up to √n reduces the number of divisions from O(n) to O(√n), making the function fast even for large numbers like 999983.

### Q2. Sieve of Eratosthenes
Implement the Sieve of Eratosthenes to find all primes up to `n`. Return them as a list. Test with `n = 50` and `n = 200`. Compare the output count to the known prime-counting values.

In [2]:
# Q2: Sieve of Eratosthenes — all primes up to n

import math

def sieve_of_eratosthenes(n):
    if n < 2:
        return []
    is_prime = [True] * (n + 1)        # assume all numbers are prime initially
    is_prime[0] = is_prime[1] = False   # 0 and 1 are not prime

    for i in range(2, math.isqrt(n) + 1):   # only need to sieve up to √n
        if is_prime[i]:
            for multiple in range(i * i, n + 1, i):  # mark multiples as not prime
                is_prime[multiple] = False             # start at i² (smaller already marked)

    return [num for num in range(n + 1) if is_prime[num]]

# Test with n = 50
primes_50 = sieve_of_eratosthenes(50)
print(f"Primes up to  50 ({len(primes_50):>3} found): {primes_50}")

# Test with n = 200
primes_200 = sieve_of_eratosthenes(200)
print(f"Primes up to 200 ({len(primes_200):>3} found): {primes_200}")

# Compare against known prime-counting values (π(n))
# π(50) = 15, π(200) = 46
print(f"\nVerification:")
print(f"  π(50)  = 15  → got {len(primes_50):>2}  {'PASS' if len(primes_50) == 15 else 'FAIL'}")
print(f"  π(200) = 46  → got {len(primes_200):>2}  {'PASS' if len(primes_200) == 46 else 'FAIL'}")

Primes up to  50 ( 15 found): [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Primes up to 200 ( 46 found): [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199]

Verification:
  π(50)  = 15  → got 15  PASS
  π(200) = 46  → got 46  PASS


### How it works

- We create a boolean list `is_prime` of size `n + 1`, with every entry set to `True` at the start.
- We immediately mark index `0` and `1` as `False` — neither is prime.
- For each number `i` from `2` to √n: if `i` is still marked prime, we mark all its **multiples** as not prime.
- We start marking from `i * i` (not `2 * i`) because smaller multiples like `2i`, `3i` … were already marked by earlier primes.
- Once the outer loop finishes, every index still `True` in the list is a prime number.
- Think of it like a strainer: we pour all numbers in and the sieve knocks out the non-primes one batch at a time.

**Key takeaway:** The Sieve of Eratosthenes is much faster than checking each number individually — it finds all primes up to n in roughly O(n log log n) time by eliminating multiples in bulk.

### Q3. GCD and LCM
Implement `gcd(a, b)` using the Euclidean algorithm (iterative, no `math.gcd`). Then implement `lcm(a, b)` using the relationship `lcm = a * b // gcd(a, b)`. Extend both to accept any number of arguments using `functools.reduce`. Verify against `math.gcd`.

In [3]:
# Q3: GCD via Euclidean algorithm and LCM via GCD relationship

import math
import functools

def gcd(a, b):
    while b:              # keep going until remainder is 0
        a, b = b, a % b  # replace (a, b) with (b, a mod b)
    return a

def lcm(a, b):
    return a * b // gcd(a, b)  # avoids overflow vs (a*b) then divide

# Extend to multiple arguments using functools.reduce
def gcd_multi(*args):
    return functools.reduce(gcd, args)  # gcd(a,b,c) = gcd(gcd(a,b), c)

def lcm_multi(*args):
    return functools.reduce(lcm, args)  # lcm(a,b,c) = lcm(lcm(a,b), c)

# --- Two-argument tests ---
pairs = [(12, 8), (100, 75), (17, 13), (0, 5), (48, 180)]
print("Two-argument verification (our gcd vs math.gcd):")
for a, b in pairs:
    our   = gcd(a, b)
    truth = math.gcd(a, b)
    flag  = "PASS" if our == truth else "FAIL"
    print(f"  gcd({a:>3}, {b:>3}) = {our:>3}  |  lcm = {lcm(a,b):>6}  |  {flag}")

# --- Multi-argument tests ---
print("\nMulti-argument:")
nums1 = [12, 18, 24]
nums2 = [4, 6, 8, 10]
print(f"  gcd_multi{tuple(nums1)} = {gcd_multi(*nums1)}"
      f"  (math: {math.gcd(*nums1)})")
print(f"  lcm_multi{tuple(nums1)} = {lcm_multi(*nums1)}")
print(f"  gcd_multi{tuple(nums2)} = {gcd_multi(*nums2)}"
      f"  (math: {math.gcd(*nums2)})")
print(f"  lcm_multi{tuple(nums2)} = {lcm_multi(*nums2)}")

Two-argument verification (our gcd vs math.gcd):
  gcd( 12,   8) =   4  |  lcm =     24  |  PASS
  gcd(100,  75) =  25  |  lcm =    300  |  PASS
  gcd( 17,  13) =   1  |  lcm =    221  |  PASS
  gcd(  0,   5) =   5  |  lcm =      0  |  PASS
  gcd( 48, 180) =  12  |  lcm =    720  |  PASS

Multi-argument:
  gcd_multi(12, 18, 24) = 6  (math: 6)
  lcm_multi(12, 18, 24) = 72
  gcd_multi(4, 6, 8, 10) = 2  (math: 2)
  lcm_multi(4, 6, 8, 10) = 120


### How it works

- **Euclidean algorithm:** at each step, replace `(a, b)` with `(b, a % b)`. When `b` reaches `0`, `a` holds the GCD. Think of it as repeatedly finding the remainder until nothing is left over.
- **LCM from GCD:** the relationship `lcm(a, b) = a * b // gcd(a, b)` comes directly from number theory. We use integer division `//` *after* dividing by GCD to avoid unnecessarily large intermediate products.
- **`functools.reduce(f, [a, b, c])`** applies `f` left-to-right: `f(f(a, b), c)`. This collapses a list of numbers into one value using any two-argument function.
- GCD and LCM are both **associative**, so chaining them with `reduce` gives the correct result for any number of arguments.
- `gcd(0, 5)` correctly returns `5` because every number divides 0 — the algorithm handles this naturally without a special case.

**Key takeaway:** `functools.reduce` is the standard way to extend any two-argument mathematical operation (GCD, LCM, sum, product) to work across an entire list.

### Q4. Factorial — Iterative and Recursive
Implement `factorial_iterative(n)` using a loop and `factorial_recursive(n)` using recursion. Both should handle `n = 0`. Compare their call behavior for large `n` — at what `n` does the recursive version hit Python's recursion limit?

In [5]:
# Q4: Factorial — iterative and recursive, plus recursion limit exploration

import sys
import math

sys.set_int_max_str_digits(50000)


def factorial_iterative(n):
    result = 1
    for i in range(2, n + 1):  # range(2, 1) is empty, so n=0 → returns 1
        result *= i
    return result

def factorial_recursive(n):
    if n == 0:                  # base case: 0! = 1 by definition
        return 1
    return n * factorial_recursive(n - 1)  # recursive case

# --- Basic correctness check against math.factorial ---
test_values = [0, 1, 5, 10, 20]
print("Correctness check:")
for n in test_values:
    it  = factorial_iterative(n)
    rec = factorial_recursive(n)
    ref = math.factorial(n)
    flag = "PASS" if it == rec == ref else "FAIL"
    print(f"  n={n:>2}  iterative={it:>20}  recursive={rec:>20}  {flag}")

# --- Find the recursion limit ---
print(f"\nPython default recursion limit: {sys.getrecursionlimit()}")

limit_n = None
for n in range(0, sys.getrecursionlimit(), 10):  # step by 10 for speed
    try:
        factorial_recursive(n)
    except RecursionError:
        limit_n = n
        break

# Narrow down to exact value
if limit_n is not None:
    for n in range(limit_n - 10, limit_n):
        try:
            factorial_recursive(n)
        except RecursionError:
            limit_n = n
            break

print(f"Recursive version crashes at n = {limit_n}")
print(f"Iterative version handles n = 10000 fine: {len(str(factorial_iterative(10000)))} digits")

Correctness check:
  n= 0  iterative=                   1  recursive=                   1  PASS
  n= 1  iterative=                   1  recursive=                   1  PASS
  n= 5  iterative=                 120  recursive=                 120  PASS
  n=10  iterative=             3628800  recursive=             3628800  PASS
  n=20  iterative= 2432902008176640000  recursive= 2432902008176640000  PASS

Python default recursion limit: 3000
Recursive version crashes at n = 2978
Iterative version handles n = 10000 fine: 35660 digits


### How it works

- **Iterative:** a simple loop multiplies `result` by every integer from `2` to `n`. Starting from `2` skips a redundant `× 1`. When `n = 0`, `range(2, 1)` is empty, so `result` stays `1` — correct by definition.
- **Recursive:** the function calls *itself* with `n - 1` until it hits the base case `n == 0`. Each call is stacked in memory until the base case unwinds them all.
- Python keeps a **call stack** in memory. Every recursive call adds one frame. The default limit is `1000` frames (`sys.getrecursionlimit()`).
- Once the stack depth exceeds that limit, Python raises a `RecursionError` to prevent a crash — so the recursive version fails somewhere around `n = 995–998` (a few frames are already used by the notebook itself).
- The iterative version uses **constant stack space** regardless of `n`, so it handles `n = 10 000` or beyond without any issue.

**Key takeaway:** Recursion is elegant and mirrors the mathematical definition, but every call consumes a stack frame — for deep problems like large factorials, the iterative version is always safer in Python.

### Q5. Fibonacci Sequence — Three Ways
Implement Fibonacci three ways:
- Iterative (loop)
- Recursive (naive)
- Using a generator (`yield`)
Generate the first 15 Fibonacci numbers with each approach and verify they match.

In [6]:
# Q5: Fibonacci — iterative, recursive, and generator

# --- Method 1: Iterative ---
def fibonacci_iterative(n):
    fibs = []
    a, b = 0, 1
    for _ in range(n):       # generate exactly n terms
        fibs.append(a)
        a, b = b, a + b      # advance: next pair is (b, a+b)
    return fibs

# --- Method 2: Recursive (naive) ---
def fib_recursive(n):
    if n == 0:               # base case 1
        return 0
    if n == 1:               # base case 2
        return 1
    return fib_recursive(n - 1) + fib_recursive(n - 2)  # two recursive calls

def fibonacci_recursive(n):
    return [fib_recursive(i) for i in range(n)]

# --- Method 3: Generator ---
def fibonacci_generator(n):
    a, b = 0, 1
    for _ in range(n):
        yield a              # pause here, return a, resume on next()
        a, b = b, a + b

# --- Generate first 15 terms with each method ---
N = 15
iter_result = fibonacci_iterative(N)
rec_result  = fibonacci_recursive(N)
gen_result  = list(fibonacci_generator(N))  # consume generator into a list

print(f"Iterative : {iter_result}")
print(f"Recursive : {rec_result}")
print(f"Generator : {gen_result}")

# --- Verify all three match ---
all_match = iter_result == rec_result == gen_result
print(f"\nAll three match: {all_match}")

# --- Show recursive call cost vs iterative ---
import time

def count_calls():
    counter = [0]
    def fib(n):
        counter[0] += 1
        if n <= 1:
            return n
        return fib(n - 1) + fib(n - 2)
    fib(N - 1)
    return counter[0]

print(f"\nRecursive calls needed for fib({N-1}): {count_calls()}")
print(f"Iterative steps needed for fib({N-1}): {N} (one per term)")

Iterative : [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]
Recursive : [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]
Generator : [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]

All three match: True

Recursive calls needed for fib(14): 1219
Iterative steps needed for fib(14): 15 (one per term)


### How it works

- **Iterative:** tracks just two variables `a` and `b`. Each loop step appends `a`, then slides the window forward: `a, b = b, a + b`. Memory use stays constant no matter how large `n` is.
- **Recursive (naive):** directly mirrors the mathematical definition — `fib(n) = fib(n-1) + fib(n-2)`. Clean to read, but each call spawns *two* more calls, so the total work grows exponentially (roughly 2ⁿ calls for `fib(n)`).
- **Generator:** uses `yield` to pause the function mid-execution and hand back one value at a time. The local variables `a` and `b` are remembered between yields — so it is as efficient as iterative but produces values lazily (on demand).
- A generator does not compute all values upfront. You can iterate over `fibonacci_generator(1_000_000)` without storing a million numbers in memory at once.
- The call-count comparison makes the recursive cost visible: `fib(14)` alone requires hundreds of redundant calls while the iterative and generator versions each take exactly 15 steps.

**Key takeaway:** All three produce identical results, but their costs differ sharply — iterative and generator are O(n), while naive recursion is O(2ⁿ) and becomes impractically slow for even moderately large `n`.